In [ ]:
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
import os

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

print("Token loaded ✅")

Token loaded ✅


In [ ]:
!pip install datasets==2.14.7

In [3]:
dataset = load_dataset(
    "liweili/c4_200m",
    split="train",
    streaming=True,
    token=hf_token
)

samples = []
for i, example in enumerate(dataset):
    samples.append(example)
    if i >= 9999:
        break

print(f"Loaded {len(samples)} samples")
print("\nFirst 3 samples:")
for s in samples[:3]:
    print(s)

Loaded 10000 samples

First 3 samples:
{'input': 'Bitcoin is for $7,094 this morning, which CoinDesk says.', 'output': 'Bitcoin goes for $7,094 this morning, according to CoinDesk.'}
{'input': 'The effect of widespread dud targets two face up attack position monsters on the field.', 'output': '1. The effect of "widespread dud" targets two face up attack position monsters on the field.'}
{'input': 'tax on sales of stores for non residents are set at 21% for 2014 and 20% in 2015 payable on sales tentatively earned from the difference of the property value some time of purchase (price differences according to working time) and theyear to which sale couples (sales costs), based on the approved annual on the base approved by law).', 'output': 'Capital Gains tax on the sale of properties for non-residents is set at 21% for 2014 and 20% in 2015 payable on profits earned on the difference of the property value between the year of purchase (purchase price plus costs) and the year of sale (sales

In [ ]:
import numpy as np

input_lengths = [len(s['input'].split()) for s in samples]
output_lengths = [len(s['output'].split()) for s in samples]
identical = sum(1 for s in samples if s['input'] == s['output'])

print(f"Input length  — mean: {np.mean(input_lengths):.1f}, max: {max(input_lengths)}, min: {min(input_lengths)}")
print(f"Output length — mean: {np.mean(output_lengths):.1f}, max: {max(output_lengths)}, min: {min(output_lengths)}")
print(f"Identical pairs (no correction needed): {identical} / {len(samples)} ({100*identical/len(samples):.1f}%)")

short = sum(1 for l in input_lengths if l <= 10)
medium = sum(1 for l in input_lengths if 11 <= l <= 20)
long = sum(1 for l in input_lengths if l > 20)
print(f"\nLength buckets:")
print(f"  Short  (≤10 tokens): {short}")
print(f"  Medium (11-20):      {medium}")
print(f"  Long   (>20):        {long}")

Input length  — mean: 21.9, max: 358, min: 3
Output length — mean: 21.8, max: 145, min: 5
Identical pairs (no correction needed): 59 / 10000 (0.6%)

Length buckets:
  Short  (≤10 tokens): 2411
  Medium (11-20):      3355
  Long   (>20):        4234


In [ ]:
def token_overlap(s1, s2):
    t1, t2 = set(s1.lower().split()), set(s2.lower().split())
    if not t1 or not t2:
        return 0
    return len(t1 & t2) / max(len(t1), len(t2))

overlaps = [token_overlap(s['input'], s['output']) for s in samples]

print(f"Token overlap (input vs output):")
print(f"  Mean:   {np.mean(overlaps):.3f}")
print(f"  Median: {np.median(overlaps):.3f}")
print(f"  Min:    {min(overlaps):.3f}")
print(f"  Max:    {max(overlaps):.3f}")

near_identical = sum(1 for o in overlaps if o > 0.9)
heavy_rewrite = sum(1 for o in overlaps if o < 0.5)
print(f"\nNear-identical (overlap > 90%): {near_identical} ({100*near_identical/len(samples):.1f}%)")
print(f"Heavy rewrites (overlap < 50%): {heavy_rewrite} ({100*heavy_rewrite/len(samples):.1f}%)")

Token overlap (input vs output):
  Mean:   0.784
  Median: 0.812
  Min:    0.000
  Max:    1.000

Near-identical (overlap > 90%): 1829 (18.3%)
Heavy rewrites (overlap < 50%): 368 (3.7%)
